# Synax Code — train model Python từ số 0

Dataset: `codeparrot/codeparrot-clean`, chỉ giữ license permissive
(Apache-2.0, MIT, BSD, ISC, Unlicense, CC0). Xem `docs/02-python.md`.

Notebook có 2 chế độ: `test` (200MB) và `full` (4GB).


In [ ]:
import os
import torch

print("torch:", torch.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("bf16:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

REPO = "/content/Synax"
if not os.path.exists(REPO):
    !git clone https://github.com/wiothemilo-lang/Synax.git {REPO}
os.chdir(REPO)
!pip install -q -e ".[data]"


In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/synax"
os.makedirs(DRIVE, exist_ok=True)
print("drive:", DRIVE)

# Tùy chọn: dùng HF token để tải nhanh hơn, tránh rate limit.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF token: loaded")
except Exception:
    print("HF token: not set (anonymous access)")


## 0. Chọn chế độ

- `MODE = "test"` → 200MB, khoảng 5–10 phút, dùng để kiểm tra.
- `MODE = "full"` → 4GB, khoảng 30–60 phút, dùng để train thật.

Hai chế độ lưu ở thư mục riêng, không ghi đè nhau.


In [ ]:
MODE = "test"  # đổi thành "full" khi chạy 4GB
MAX_BYTES = 200_000_000 if MODE == "test" else 4_000_000_000
DATA = f"{DRIVE}/python/{MODE}"
CORPUS = f"{DATA}/clean/python.jsonl"
TOKENIZER = f"{DRIVE}/tokenizer/sp32-code-{MODE}"
TRAIN_BIN = f"{DATA}/tokenized/train.bin"
VAL_BIN = f"{DATA}/tokenized/val.bin"
OUT = f"{DRIVE}/checkpoints/60m-code-{MODE}"
TMP = "/content/synax-tmp"
from pathlib import Path

for path in (
    Path(CORPUS).parent,
    Path(TOKENIZER).parent,
    Path(TRAIN_BIN).parent,
    Path(OUT),
    Path(TMP),
):
    path.mkdir(parents=True, exist_ok=True)
print("mode:", MODE, "max_bytes:", MAX_BYTES)
print("corpus:", CORPUS)
print("output:", OUT)


## 1. Chuẩn bị corpus Python

Stream dataset, lọc license + AST + dedupe, ghi JSONL vào Drive.
Nếu file đã tồn tại, cell sẽ bỏ qua.


In [ ]:
if not Path(CORPUS).exists():
    !python -m synax.data.code --output {CORPUS} --max-bytes {MAX_BYTES} --progress-every 20000
else:
    print("corpus exists:", CORPUS, Path(CORPUS).stat().st_size, "bytes")


In [ ]:
if not Path(TOKENIZER + ".model").exists():
    !python -m synax.data.tokenizer train --input {CORPUS} --output-prefix {TOKENIZER} --vocab-size 32000 --code --tmp-dir {TMP}
else:
    print("tokenizer exists:", TOKENIZER + ".model")


In [ ]:
if not Path(TRAIN_BIN).exists():
    !python -m synax.data.tokenizer encode --input {CORPUS} --tokenizer {TOKENIZER}.model --output {TRAIN_BIN} --val-output {VAL_BIN} --val-ratio 0.005 --code
else:
    print("shards exist")


## 2. Train

- `test`: 200 step, khoảng 10–20 phút trên GPU.
- `full`: chạy đến 31.000 step, có thể ngắt và `--resume auto`.


In [ ]:
if MODE == "test":
    !python -m synax.train.pretrain --config configs/model-60m-code.yaml --train-bin {TRAIN_BIN} --val-bin {VAL_BIN} --tokenizer {TOKENIZER}.model --out-dir {OUT} --steps 200 --warmup-steps 20
else:
    !python -m synax.train.pretrain --config configs/model-60m-code.yaml --train-bin {TRAIN_BIN} --val-bin {VAL_BIN} --tokenizer {TOKENIZER}.model --out-dir {OUT} --resume auto


In [ ]:
latest = (Path(OUT) / "latest.txt").read_text().strip()
print("checkpoint:", Path(OUT) / latest)

!python -m synax.inference.generate --checkpoint {OUT}/{latest} --tokenizer {TOKENIZER}.model --prompt "def fibonacci(n):" --max-new-tokens 120


## Ghi chú

- 200MB chỉ để test pipeline; model thật cần ~4GB text (~1B token).
- Checkpoint giữ 2 bản mới nhất trong Drive.
- Không commit data/checkpoint vào git.
- Khi chuyển sang `full`, chỉ cần đổi `MODE` rồi chạy lại từ cell 0.
